In [1]:
from malab import *
from os.path import join
from glob import glob
from malab.dsfit import *
import os 
from numpy import *
from tempfile import TemporaryFile
%matplotlib inline
from matplotlib.pyplot import *
from scipy.signal import savgol_filter
import scipy.constants as constants

font = {'weight' : 'normal',
        'size'   : 18}

matplotlib.rc('font', **font)

No module named 'visa'
Warning VISA library import failed
Warning serial library import failed.
Could not load InstrumentManagerWindow


In [2]:
from malab.instruments.voltsource import *
from malab.instruments.instrumenttypes import *
from time import sleep

In [3]:
dac = AD5780(address='192.168.1.200') # 65 for Alice
print((dac.get_id()))

# #be careful - initializing will reset all channel back to 0 directly
#dac.initialize()
time.sleep(1)

#V/s or mA/s
dac_rate = 0.05

Welcome to the Telnet server!
Type HELP for instructions.
AD5780



In [4]:
for i in range(1, 3):
    dac.initialize(i)

In [5]:
dac.get_voltage(1)

'131071\r\n'

In [26]:
def ramp(channel, voltage, speed):
    """Ramp to voltage with speed in (V/S)"""
    bitcode = int((voltage + 10.0)*13107.2)
    print('target', bitcode)
    time.sleep(dac.query_sleep)
    if bitcode < 0 or bitcode > 262143:
        print('ERROR: voltage out of range')
        return str(bitcode)
    step_size = 10 # in bits, about 0.7mV out of +-10V
    step_time = int(step_size * 76.2939453 / speed)
    if step_time == 0:
        step_time = 1
    endbit = dac.query('RAMP %d %d %d %d' % (channel, bitcode, step_size, step_time))
    print('RAMP %d %d %d %d' % (channel, bitcode, step_size, step_time))
    #print('end of ramp -', endbit)
    time.sleep(dac.query_sleep)
    return endbit

In [18]:
V_PER_BIT = 20.0 / 2**18   # 76.2939 uV per bit (18-bit, ±10 V)

def ramp(channel, voltage, speed):
    """Ramp channel to voltage at `speed` (V/s). Firmware step_time is in ms."""
    bitcode = int(round((voltage + 10.0) / V_PER_BIT))
    print('target', bitcode)
    time.sleep(dac.query_sleep)
    if bitcode < 0 or bitcode > 2**18 - 1:
        print('ERROR: voltage out of range')
        return str(bitcode)

    # aim for ~10-bit (0.76 mV) steps; floor at the firmware's 1 ms minimum
    step_time = max(1, round(10 * V_PER_BIT * 1e3 / speed))          # ms
    # re-solve step_size for that step_time so the rate is honored
    step_size = max(1, round(speed * step_time * 1e-3 / V_PER_BIT))  # bits

    actual = step_size * V_PER_BIT / (step_time * 1e-3)
    print('ramp rate: requested %.4g V/s, actual %.4g V/s' % (speed, actual))

    cmd = 'RAMP %d %d %d %d' % (channel, bitcode, step_size, step_time)
    endbit = dac.query(cmd)
    print(cmd)
    print('end of ramp -', endbit)
    time.sleep(dac.query_sleep)
    return endbit

In [38]:
dac.set_voltage(1, 2)
#dac.get_voltage(1)

set target 157286


'SET command received with parameters: 1 157286\r\n'

In [7]:
def digit_to_curr(x):
    return 20*x/(2**18)-10

# This is the unity gain of the difference between the input voltages to the DAC. 
# AD5780 DAC is 18-bit so the output is 2^18 divisions of the differential input voltage. 
# The output voltage is then given by V_out = (V_pos - V_neg) * (DAC_code / 2^18)  - V_neg
# where V_pos and V_neg are the positive and negative reference voltages to the DAC, and DAC_code is the digital code sent to the DAC. 
# In our case, we have V_pos = 10V and V_neg = -10V, so the output voltage is given by V_out = 20 * (DAC_code / 2^18) - 10. 
# Therefore, the current output from the DAC is given by I_out = V_out / R, where R is the resistance in the current output stage. 
# We have an R = 1kOhm resistor on each DAC, and the on-chip flux line itself is a short circuit on the other side of the resistor. 
# Other resistances are negligible compared to the 1kOhm resistor, so we can approximate the current output as I_out = V_out / 1000.
# we have I_out = (20 * (DAC_code / 2^18) - 10) / 1000 = 20 * DAC_code / (2^18 * 1000) - 10 / 1000 = 20 * DAC_code / (2^18 * 1000) - 0.01 A
# Therefore, the current output from the DAC is the same value as the voltage output, but in units of mA instead of V.
# Since your voltage is relative to the negative reference voltage, the zero voltage point corresponds to a DAC code of 131072, which gives an output current of 0 mA.

In [ ]:
round(digit_to_curr(int(dac.get_voltage(1))), 3)

0.0

In [29]:
ramp(int(1), 1.000, 0.1) # voltage units here simple translate to current units by the digit_to_curr function, so this is ramping to 0.1 mA at a rate of 0.05 mA/s

target 144179
RAMP 1 144179 10 7629


'RAMP command received with parameters: 1 131072 10 7629\r\n'

# DC flux crosstalk matrix [calibrated around 2023/09]

In [7]:
#### DC flux crosstalk matrix ####

alpha = np.array([[-242.31714841,   45.89614727,  -25.29120955,  -50.01008227],
       [  60.45699591, -190.76334318,  -11.23645568,  -49.74869705],
       [  64.83775   ,   69.87982455, -261.90374136,  -75.15842114],
       [  48.19585591,   47.83957455,   31.68753295, -313.47893068]])

alpha_inv = np.linalg.inv(alpha)

# Turn back to zero [only if you need]

In [104]:
for ii in range(4):
    end_current = 0e-3
    print('\n')
    print('DAC CH: ',ii+1)
    print("Returning DAC to (%.3f) V" % (end_current))
    dac.get_voltage(int(ii + 1))
    dac.get_voltage(int(ii + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(ii+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(ii+1), end_current, dac_rate)
    time.sleep(abs(dac_old - end_current) / dac_rate + 30)
    time.sleep(0.2)
    dac.get_voltage(int(ii+1))



DAC CH:  1
Returning DAC to (0.000) V
old reading =  6.5480804443359375
target 131072
end of ramp - None


DAC CH:  2
Returning DAC to (0.000) V
old reading =  8.528518676757812
target 131072
end of ramp - None


DAC CH:  3
Returning DAC to (0.000) V
old reading =  5.36590576171875
target 131072
end of ramp - None


DAC CH:  4
Returning DAC to (0.000) V
old reading =  5.978240966796875
target 131072
end of ramp - None


# Read the current DAC values (05/08/2025 to check Qihao's setting)

In [7]:
for i_flux in range(4):
    reading = dac.get_voltage(int(i_flux + 1))
    print(f'DAC{i_flux+1} Reading = ', dac.get_voltage(int(i_flux + 1)))

DAC1 Reading =  216899

DAC2 Reading =  242857

DAC3 Reading =  201400

DAC4 Reading =  209430



In [8]:
for i_flux in range(4):
    reading = dac.get_voltage(int(i_flux + 1))
    print(f'DAC{i_flux+1} Reading = ', dac.get_voltage(int(i_flux + 1)))

DAC1 Reading =  216899

DAC2 Reading =  242857

DAC3 Reading =  201400

DAC4 Reading =  209430



# [local for Qihao's 2Q stab] - Q3Q4 around 4.47GHz


- The qubits are aligned around 4.4GHz with default 'flux_current'
- Using d_freq to tune the array to target location, unit of MHz
- Update the DAC setting to 'new_flux_current'

In [11]:
flux_current = array([6.5482333 , 8.52866189, 5.36523859, 5.97820194])
# d_freq =  -500*np.array([1, 1, 0, 0])

# ## Here the first array is measured Ramsey frequency and subtract the targeting ramsey frequency [5MHz in this example]
#d_freq = - np.array([0, 0, 62.067372 , 64.1827585]) + np.array([0, 0, 70, 70])
d_freq =  array([5        , 5       ,         4.67258244, 5.17774583]) - np.array([5, 5 ,5 - 0.2, 5 + 0.2])## in MHz

print('Tune qubit by ??? MHz')
print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
#new_flux_current = flux_current
new_flux_current

Tune qubit by ??? MHz
[ 0.          0.         -0.12741756 -0.02225417]


array([6.54815672, 8.52858909, 5.36566066, 5.97829271])

In [12]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (6.548) mA
old reading =  6.5480804443359375
target 216899
end of ramp - RAMP
Driving DAC at (8.529) mA
old reading =  8.528518676757812
target 242857
end of ramp - RAMP
Driving DAC at (5.366) mA
old reading =  5.3656005859375
target 201400
end of ramp - RAMP
Driving DAC at (5.978) mA
old reading =  5.978240966796875
target 209430
end of ramp - RAMP



# Q3,Q4 collective decay at 4.6GHz

In [8]:
for i_flux in range(4):
    reading = dac.get_voltage(int(i_flux + 1))
    print(f'DAC{i_flux+1} Reading = ', dac.get_voltage(int(i_flux + 1)))

DAC1 Reading =  176448

DAC2 Reading =  189909

DAC3 Reading =  178571

DAC4 Reading =  189326



In [54]:
flux_current = array([5.79742783, 7.49915088, 4.70851481, 5.33445282])
d_freq = np.array([0, 0, 5.11144575, 5.1536498]) - np.array([0.       ,   0.       , 5, 5])
# d_freq = np.array([0, 0, 100, 100]) - np.array([0.      ,   0.      , 99.844176, 98.977251])
# d_freq = np.array([-500, -500, 0, 0])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[0.         0.         0.11144575 0.1536498 ]


array([5.79757869, 7.49933103, 4.70830685, 5.33399234])

In [55]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (5.798) mA
old reading =  5.7250213623046875
target 207062
end of ramp - None
Driving DAC at (7.499) mA
old reading =  7.398681640625
target 229367
end of ramp - None
Driving DAC at (4.708) mA
old reading =  4.5691680908203125
target 192784
end of ramp - None
Driving DAC at (5.334) mA
old reading =  5.6632232666015625
target 200985
end of ramp - None


# Q3 4600MHz and Q4 4420MHz

In [36]:
flux_current = array([5.71052452, 7.37846569, 4.5411363 , 5.72942077 ])
d_freq = np.array([0, 0, 5.05504162, 4.87046643]) - np.array([0.       ,   0.       , 5, 5])
# d_freq = np.array([0, 0, 100, -50]) - np.array([0.      ,   0.      ,100.146535, -76.359322])
# d_freq = np.array([0, 0, 0, 30])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[ 0.          0.          0.05504162 -0.12953357]


array([5.71047061, 7.37837636, 4.54078681, 5.72977673])

In [44]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (5.729) mA
old reading =  5.71044921875
target 206166
end of ramp - RAMP
Driving DAC at (7.405) mA
old reading =  7.3783111572265625
target 228124
end of ramp - RAMP
Driving DAC at (4.577) mA
old reading =  4.5407867431640625
target 191065
end of ramp - RAMP
Driving DAC at (5.644) mA
old reading =  5.7297515869140625
target 205051
end of ramp - None


# Q3 4600MHz and Q4 4450MHz [use this one]

In [52]:
flux_current = array([5.72485078, 7.39839898, 4.56937425, 5.66401779])
d_freq = np.array([0, 0, 5.13415825, 5.26187468]) - np.array([0.       ,   0.       , 5, 5])
# d_freq = np.array([0, 0, 100, -50]) - np.array([0.      ,   0.      ,99.925722, -52.231778])
# d_freq = np.array([0, 0, 0, 30])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[0.         0.         0.13415825 0.26187468]


array([5.72508047, 7.3986826 , 4.5692163 , 5.66324504])

In [53]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (5.725) mA
old reading =  5.72479248046875
target 206111
end of ramp - RAMP

Driving DAC at (7.399) mA
old reading =  7.39837646484375
target 228048
end of ramp - RAMP

Driving DAC at (4.569) mA
old reading =  4.5693206787109375
target 190961
end of ramp - RAMP
Driving DAC at (5.663) mA
old reading =  5.6639862060546875
target 205301
end of ramp - RAMP


# [2025/05/19] Align lattice at 4600MHz

In [26]:
flux_current = array([3.46168425, 4.49184688, 3.62295412, 4.44621168])
d_freq = array([4.89703272, 4.96350556, 4.92613543, 4.9787895]) - np.ones(4)*(5)## in MHz
#d_freq = -80 - np.array([-87.0332915, -85.4477295, -88.104658 , -89.4446165])
#d_freq = 100*np.ones(4) - np.array([100.4626105, 100.0457305, 100.254312 ,  97.0912305])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[-0.10296728 -0.03649444 -0.07386457 -0.0212105 ]


array([3.46207256, 4.49208581, 3.62333789, 4.4464143 ])

In [27]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (3.462) mA
old reading =  3.46160888671875
target 176450
end of ramp - RAMP

Driving DAC at (4.492) mA
old reading =  4.4918060302734375
target 189950
end of ramp - RAMP

Driving DAC at (3.623) mA
old reading =  3.622894287109375
target 178563
end of ramp - RAMP

Driving DAC at (4.446) mA
old reading =  4.4461822509765625
target 189352
end of ramp - RAMP



# [2026/06/08] Align lattice at [-215.95, 99.46, -66.04, 285.96]MHz

In [26]:
flux_current = array([4.53928823, 4.81384694, 4.52102419, 4.27739342])
d_freq = array([4.96748714, 4.99737513, 5.01592598, 4.99338521]) - np.ones(4)*(5)## in MHz
#d_freq =  np.array([-215.957, 99.458,  -66.039,  285.963]) - np.array([-208.123551,101.262748,-55.85333,279.096377])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[-0.03251286 -0.00262487  0.01592598 -0.00661479]


array([4.53942457, 4.81389347, 4.52099625, 4.27743976])

In [27]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (4.539) mA
old reading =  4.5392608642578125
target 190571
end of ramp - RAMP

Driving DAC at (4.814) mA
old reading =  4.8138427734375
target 194168
end of ramp - RAMP
Driving DAC at (4.521) mA
old reading =  4.5209503173828125
target 190329
end of ramp - RAMP
Driving DAC at (4.277) mA
old reading =  4.27734375
target 187137
end of ramp - RAMP



# [2025/06/15] Align lattice at [-300, 450, 450, 450] MHz

In [38]:
flux_current = array([4.71943176, 4.40460633, 3.57186305, 4.30014136])
#d_freq = array([5.05201464, 4.99099102, 5.047877  , 4.9360707]) - np.ones(4)*(5)## in MHz
d_freq =  np.array([-300, 400, 400, 400]) - np.array([-257.98645,   221.877364,  220.151453,  219.582538])

print(d_freq)

new_flux_current = flux_current+ np.dot(alpha_inv, d_freq)
new_flux_current

[-42.01355  178.122636 179.848547 180.417462]


array([4.9772923 , 3.76829538, 2.97809955, 3.60712701])

In [39]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (4.977) mA
old reading =  2.110443115234375
target 196310
end of ramp - None
Driving DAC at (3.768) mA
old reading =  2.6508331298828125
target 180463
end of ramp - None
Driving DAC at (2.978) mA
old reading =  1.6484832763671875
target 170106
end of ramp - None
Driving DAC at (3.607) mA
old reading =  4.7875213623046875
target 178351
end of ramp - None


# [2025/06/15] Align lattice at [300, -300, 400, 100] MHz

In [42]:
flux_current = array([2.87665991, 6.40989596, 3.13985874, 3.71240794])
#d_freq = array([5.05212561, 4.93615176, 4.9950412 , 4.94804342]) - np.ones(4)*(5)## in MHz
#d_freq = -80 - np.array([-87.0332915, -85.4477295, -88.104658 , -89.4446165])
d_freq =  np.array([400, -200,  400,  400]) - np.array([434.372332, -302.938918,  437.546856,  437.784325])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[-34.372332 102.938918 -37.546856 -37.784325]


array([2.90795623, 5.86946523, 3.13462249, 3.7547482 ])

In [43]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (2.908) mA
old reading =  2.9079437255859375
target 169187
end of ramp - RAMP
Driving DAC at (5.869) mA
old reading =  5.86944580078125
target 208004
end of ramp - RAMP
Driving DAC at (3.135) mA
old reading =  3.134613037109375
target 172158
end of ramp - RAMP
Driving DAC at (3.755) mA
old reading =  3.754730224609375
target 180286
end of ramp - RAMP


# [2025/06/15] Align lattice at [100,400, -300, 300] MHz

In [44]:
flux_current = array([2.91124915, 3.84083886, 4.57640781, 4.03764921])
#d_freq = array([5.05212561, 4.93615176, 4.9950412 , 4.94804342]) - np.ones(4)*(5)## in MHz
#d_freq = -80 - np.array([-87.0332915, -85.4477295, -88.104658 , -89.4446165])
d_freq =  np.array([400, 400,  -300,  400]) - np.array([221.924791,  221.924639, -258.567534,  227.761172])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[178.075209 178.075361 -41.432466 172.238828]


array([2.16853877, 2.88697275, 4.52979924, 3.22373853])

In [45]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (2.169) mA
old reading =  2.9079437255859375
target 159495
end of ramp - None
Driving DAC at (2.887) mA
old reading =  5.86944580078125
target 168912
end of ramp - None
Driving DAC at (4.530) mA
old reading =  3.134613037109375
target 190444
end of ramp - None
Driving DAC at (3.224) mA
old reading =  3.754730224609375
target 173326
end of ramp - None


# [2025/06/15] Align lattice at [400, 400, 400, -300] MHz

In [46]:
flux_current = array([2.85791236, 3.66154809, 2.68692828, 5.08709534])
#d_freq = array([5.05212561, 4.93615176, 4.9950412 , 4.94804342]) - np.ones(4)*(5)## in MHz
#d_freq = -80 - np.array([-87.0332915, -85.4477295, -88.104658 , -89.4446165])
d_freq =  np.array([400, 400,  400,  -300]) - np.array([224.030809,  225.816246,  224.61179,  -276.621983])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[175.969191 174.183754 175.38821  -23.378017]


array([2.11049129, 2.6508674 , 1.64852194, 4.78755479])

In [47]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (2.110) mA
old reading =  2.1685028076171875
target 158734
end of ramp - None
Driving DAC at (2.651) mA
old reading =  2.886962890625
target 165817
end of ramp - None
Driving DAC at (1.649) mA
old reading =  4.52972412109375
target 152679
end of ramp - None
Driving DAC at (4.788) mA
old reading =  3.223724365234375
target 193823
end of ramp - None


# Align Q0 at the Upper Sweet Spot Point (YOKO = 4mA)

In [28]:
flux_current = array([-9.18237, -2.09328173, -2.92864609, -1.96301821])

new_flux_current = flux_current
new_flux_current

array([-9.18237   , -2.09328173, -2.92864609, -1.96301821])

In [29]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (-9.182) mA
old reading =  4.977264404296875
target 10716
end of ramp - None
Driving DAC at (-2.093) mA
old reading =  3.7682342529296875
target 103634
end of ramp - None
Driving DAC at (-2.929) mA
old reading =  2.978057861328125
target 92685
end of ramp - None
Driving DAC at (-1.963) mA
old reading =  3.6071014404296875
target 105342
end of ramp - None


# Align Q1 at the Upper Sweet Spot Point (YOKO = 3mA)

In [30]:
flux_current = array([-1.03984511, -8.68030, -2.92253593, -1.84583146])

new_flux_current = flux_current
new_flux_current


array([-1.03984511, -8.6803    , -2.92253593, -1.84583146])

In [31]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (-1.040) mA
old reading =  -9.18243408203125
target 117442
end of ramp - None
Driving DAC at (-8.680) mA
old reading =  -2.093353271484375
target 17297
end of ramp - None
Driving DAC at (-2.923) mA
old reading =  -2.9286956787109375
target 92765
end of ramp - RAMP
Driving DAC at (-1.846) mA
old reading =  -1.963043212890625
target 106878
end of ramp - None


# Align Q2 at the Upper Sweet Spot Point (YOKO = 3mA)

In [32]:
flux_current = array([1.08195793,  0.92348611, -7.47468545342583 , -0.47603542])

new_flux_current = flux_current
new_flux_current

array([ 1.08195793,  0.92348611, -7.47468545, -0.47603542])

In [33]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (1.082) mA
old reading =  -1.039886474609375
target 145253
end of ramp - None
Driving DAC at (0.923) mA
old reading =  -8.680343627929688
target 143176
end of ramp - None
Driving DAC at (-7.475) mA
old reading =  -2.9225921630859375
target 33099
end of ramp - None
Driving DAC at (-0.476) mA
old reading =  -1.845855712890625
target 124832
end of ramp - None


# Align Q3 at the Upper Sweet Spot Point (YOKO = 3mA)

In [34]:
flux_current = array([1.22361369,  1.64634129,  1.45795599, -4.74508])

new_flux_current = flux_current
new_flux_current

array([ 1.22361369,  1.64634129,  1.45795599, -4.74508   ])

In [35]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (1.224) mA
old reading =  1.0819244384765625
target 147110
end of ramp - None
Driving DAC at (1.646) mA
old reading =  0.9234619140625
target 152650
end of ramp - None
Driving DAC at (1.458) mA
old reading =  -7.4747467041015625
target 150181
end of ramp - None
Driving DAC at (-4.745) mA
old reading =  -0.47607421875
target 68877
end of ramp - None


# New Readout Location - 1

In [65]:
flux_current = array([4.45416104, 3.71654775, 4.55432434, 4.48850497])
d_freq = array([4.98682078, 4.98637726, 4.98760564, 4.92548694]) - np.ones(4)*(5)## in MHz
#d_freq =  np.array([-260.0583, 329.2519, -189.8211,  130.9061]) - np.array([-255.695391,  323.020968, -180.609709,  130.827882])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[-0.01317922 -0.01362274 -0.01239436 -0.07451306]


array([4.45417007, 3.71656051, 4.55430859, 4.48874441])

In [66]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (4.454) mA
old reading =  4.4541168212890625
target 189453
end of ramp - RAMP
Driving DAC at (3.717) mA
old reading =  3.7165069580078125
target 179785
end of ramp - RAMP
Driving DAC at (4.554) mA
old reading =  4.554290771484375
target 190766
end of ramp - RAMP
Driving DAC at (4.489) mA
old reading =  4.4884490966796875
target 189906
end of ramp - RAMP



# New Readout Location - 2 

In [93]:
flux_current = array([2.87613912, 5.949814  , 4.40719082, 4.62754703])
d_freq = array([5.0440391 , 4.96838014, 4.94852948, 4.97832018]) - np.ones(4)*(5)## in MHz
#d_freq =  np.array([315.8585, -272.0106278,  -95.2030,  107.6588]) - np.array([309.169851, -268.5657,    -87.187236,  108.85238 ])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[ 0.0440391  -0.03161986 -0.05147052 -0.02167982]


array([2.87594283, 5.94989153, 4.40734051, 4.62761297])

In [94]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (2.876) mA
old reading =  2.876129150390625
target 168767
end of ramp - RAMP

Driving DAC at (5.950) mA
old reading =  5.9497833251953125
target 209058
end of ramp - RAMP

Driving DAC at (4.407) mA
old reading =  4.4071197509765625
target 188839
end of ramp - RAMP

Driving DAC at (4.628) mA
old reading =  4.627532958984375
target 191727
end of ramp - RAMP



# New Readout Location - 3 [-214.8880,  98.3041,  281.1303,  -68.909]

In [58]:
flux_current = array([4.50705228, 4.69644858, 3.20898716, 5.00369155])
#d_freq = array([4.93711057, 4.79525655, 4.96333581, 4.46305792]) - np.ones(4)*(5)## in MHz
d_freq =  np.array([-214.8880,  98.3041,  381.1303,  -68.909]) - np.array([-207.587367,   99.866118,  275.42214,   -53.771638])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[-0.06288943 -0.20474345 -0.03666419 -0.53694208]


array([4.50708141, 4.69707535, 3.20878713, 5.00548431])

In [59]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (4.507) mA
old reading =  4.506988525390625
target 190147
end of ramp - RAMP

Driving DAC at (4.697) mA
old reading =  4.6964263916015625
target 192637
end of ramp - RAMP

Driving DAC at (3.209) mA
old reading =  3.20892333984375
target 173130
end of ramp - RAMP

Driving DAC at (5.005) mA
old reading =  5.003662109375
target 196679
end of ramp - RAMP


# New Readout Location - 4 [-259.546,  98.0139,  281.2770,  -68.8736]

In [93]:
flux_current = array([4.66748879, 4.73595407, 3.2479055 , 5.03833379])
d_freq = array([5.06291449, 5.08872455, 5.0380958 , 5.12407452]) - np.ones(4)*(5)## in MHz
#d_freq =  np.array([-259.546,  98.0139,  281.2770,  -68.8736]) - np.array([-260.133907,   97.495001,  281.717802,  -73.470929])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[0.06291449 0.08872455 0.0380958  0.12407452]


array([4.66727552, 4.73556222, 3.24774743, 5.03782943])

In [94]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (4.667) mA
old reading =  4.6674346923828125
target 192246
end of ramp - RAMP
Driving DAC at (4.736) mA
old reading =  4.7359466552734375
target 193141
end of ramp - RAMP

Driving DAC at (3.248) mA
old reading =  3.247833251953125
target 173640
end of ramp - RAMP
Driving DAC at (5.038) mA
old reading =  5.038299560546875
target 197103
end of ramp - RAMP


# New Readout Location - 4 [-259.546,  98.0139,  321.2770,  -68.8736]

In [67]:
flux_current = array([4.871836  , 5.60099559, 3.27689937, 4.80728201])
d_freq = array([5.0141626 , 5.24346788, 4.98497619, 4.89952749]) - np.ones(4)*(5)## in MHz
#d_freq =  np.array([-259.546,  -68.0139,  381.2770,  90]) - np.array([-259.300822,  -67.834891,  393.766638,   90.595468])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[ 0.0141626   0.24346788 -0.01502381 -0.10047251]


array([4.87155479, 5.59964424, 3.27651725, 4.80731443])

In [68]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (4.872) mA
old reading =  4.871826171875
target 194924
end of ramp - RAMP

Driving DAC at (5.600) mA
old reading =  5.6009674072265625
target 204467
end of ramp - RAMP

Driving DAC at (3.277) mA
old reading =  3.276824951171875
target 174017
end of ramp - RAMP

Driving DAC at (4.807) mA
old reading =  4.807281494140625
target 194082
end of ramp - RAMP



# New Readout Location - 5 [ 279.546,  -98.0139,  -221.2770,  68.8736]

In [60]:
flux_current = array([2.99972369, 5.0564418 , 4.63025178, 4.55897833])
d_freq = array([6.64792134, 5.75875897, 5.56847638, 5.44224363]) - np.ones(4)*(5)## in MHz
#d_freq =  np.array([230.546,  -80.0139,  -221.2770,  100]) - np.array([240.546,  -79.430957, -220.79022,   100.04274])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[1.64792134 0.75875897 0.56847638 0.44224363]


array([2.99316103, 5.05155711, 4.62617968, 4.55540152])

In [61]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (2.993) mA
old reading =  2.9996490478515625
target 170303
end of ramp - RAMP

Driving DAC at (5.052) mA
old reading =  5.0563812255859375
target 197283
end of ramp - RAMP

Driving DAC at (4.626) mA
old reading =  4.6302032470703125
target 191708
end of ramp - RAMP

Driving DAC at (4.555) mA
old reading =  4.5589447021484375
target 190780
end of ramp - RAMP



## New Readout Location - 6 [ 390.546,  -98.0139,  -221.2770,  68.8736]

In [22]:
flux_current = array([2.46752201, 4.93552933, 4.40301512, 4.43995776])
d_freq = array([5.04771095, 4.95500313, 4.91160276, 4.96450437]) - np.ones(4)*(5)## in MHz
#d_freq =  np.array([390.546,  -80.0139,  -190.81062,  100]) - np.array([390.509486,  -80.775742, -220.380238,   100])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[ 0.04771095 -0.04499687 -0.08839724 -0.03549563]


array([2.46729259, 4.93564407, 4.40329106, 4.44008112])

In [23]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (2.467) mA
old reading =  2.467498779296875
target 163411
end of ramp - RAMP
Driving DAC at (4.936) mA
old reading =  4.935455322265625
target 195764
end of ramp - RAMP

Driving DAC at (4.403) mA
old reading =  4.4029998779296875
target 188786
end of ramp - RAMP

Driving DAC at (4.440) mA
old reading =  4.4399261474609375
target 189269
end of ramp - RAMP



# New Readout Location -7

In [10]:
flux_current = array([2.3296722 , 4.80718608, 5.3000128 , 4.52030666])
d_freq = array([5.10576854, 4.96467173, 4.98020048, 4.93475267]) - np.ones(4)*(5)## in MHz
#d_freq =  np.array([390.546,  -80.0139,  -450.81062,  85]) - np.array([390.940556,  -79.842925, -449.78658,    84.804096])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[ 0.10576854 -0.03532827 -0.01979952 -0.06524733]


array([2.32921791, 4.80719694, 5.29994076, 4.52043933])

In [11]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (2.329) mA
old reading =  2.3296356201171875
target 161601
end of ramp - RAMP
Driving DAC at (4.807) mA
old reading =  4.80712890625
target 194080
end of ramp - RAMP

Driving DAC at (5.300) mA
old reading =  5.29998779296875
target 200539
end of ramp - RAMP
Driving DAC at (4.520) mA
old reading =  4.520263671875
target 190322
end of ramp - RAMP



# New Readout Location - Two LOs ([4716.997137 4385.91366  4859.238394 4545.526257])

In [62]:
flux_current = array([3.27038385, 5.38544375, 2.95000972, 4.62360251])
d_freq = array([5.04627857, 5.04653253, 5.04491928, 4.9400044]) - np.ones(4)*(5)## in MHz
#d_freq =  np.array([300-83.1483, -113.7588,  359.6907, 43.8390]) - np.array([300-90.560375, -110.843361,   349.205123,   47.786936])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[ 0.04627857  0.04653253  0.04491928 -0.0599956 ]


array([3.27015257, 5.38512701, 2.9496753 , 4.6236762 ])

In [63]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (3.270) mA
old reading =  3.2703399658203125
target 173934
end of ramp - RAMP

Driving DAC at (5.385) mA
old reading =  5.38543701171875
target 201655
end of ramp - RAMP

Driving DAC at (2.950) mA
old reading =  2.949981689453125
target 169733
end of ramp - RAMP

Driving DAC at (4.624) mA
old reading =  4.623565673828125
target 191675
end of ramp - RAMP



# New Readout Location - Two LOs ([4716.860742 4336.924451 4859.770381 4521.35337 ])

In [84]:
flux_current = array([3.28757311, 5.60287047, 2.98333541, 4.71620907])
d_freq = array([5.0055345 , 5.02858071, 5.06957767, 4.9264291]) - np.ones(4)*(5)## in MHz
#d_freq = -80 - np.array([-87.0332915, -85.4477295, -88.104658 , -89.4446165])
#d_freq =  np.array([300-83.2058, -164.3192,  359.7034, 21.3986]) - np.array([300-90.100833, -161.55611,    349.56922,    28.658235])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[ 0.0055345   0.02858071  0.06957767 -0.0735709 ]


array([3.28752097, 5.60268433, 2.98296113, 4.71636951])

In [85]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (3.288) mA
old reading =  3.287506103515625
target 174162
end of ramp - RAMP

Driving DAC at (5.603) mA
old reading =  5.6027984619140625
target 204507
end of ramp - RAMP
Driving DAC at (2.983) mA
old reading =  2.9833221435546875
target 170170
end of ramp - RAMP
Driving DAC at (4.716) mA
old reading =  4.7161865234375
target 192890
end of ramp - RAMP



# New Readout Location - Two LOs ([4717.025501 4237.880629 4859.56212  4375.270309])

In [86]:
flux_current = array([3.24985036, 5.77538084, 2.90946566, 5.08912811])
d_freq = array([5.00753793, 4.99469821, 5.02134406, 4.97111352]) - np.ones(4)*(5)## in MHz
#d_freq = -80 - np.array([-87.0332915, -85.4477295, -88.104658 , -89.4446165])
#d_freq =  np.array([300-83.0314, -220.7223,  359.8134, -124.4942]) - np.array([300-81.401586,  -220.678037,    350.158907, -108.250411])

print(d_freq)

new_flux_current = flux_current + np.dot(alpha_inv, d_freq)
new_flux_current

[ 0.00753793 -0.00530179  0.02134406 -0.02888648]


array([3.24981578, 5.77538435, 2.90935469, 5.08920426])

In [87]:
pt = new_flux_current

for i_flux in range(4):

    ## force DAC to drive at the fluxInd value
    print("Driving DAC at (%.3f) mA" % (pt[i_flux]))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac_old = digit_to_curr(int(dac.get_voltage(int(i_flux+1))[:-2]))
    print('old reading = ', dac_old)
    dac.ramp(int(i_flux+1),pt[i_flux],dac_rate)
    time.sleep(1.5*abs(dac_old - pt[i_flux]) / dac_rate + 5)
    time.sleep(5.0)
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux + 1))
    dac.get_voltage(int(i_flux+1)) # to clear the readout queue

Driving DAC at (3.250) mA
old reading =  3.24981689453125
target 173667
end of ramp - RAMP
Driving DAC at (5.775) mA
old reading =  5.7753753662109375
target 206771
end of ramp - RAMP

Driving DAC at (2.909) mA
old reading =  2.909393310546875
target 169205
end of ramp - RAMP

Driving DAC at (5.089) mA
old reading =  5.089111328125
target 197777
end of ramp - RAMP

